# Correlation Analysis: External Factors

This notebook analyzes correlations between Ulaanbaatar weather and external factors:
- Global temperature / climate change
- Global population growth
- Russia & China economic/political indicators
- CO2 emissions

## Prerequisites:
You need to obtain external data from various sources (see README for data sources).
Save external data files to: `data/external/`

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

sys.path.append('../')

from config.config import *
from src.analysis.correlation_analysis import *
from src.visualization.plots import *

pd.set_option('display.max_columns', None)
%matplotlib inline

print("Libraries loaded!")

In [ ]:
# Load Ulaanbaatar processed data
df_yearly = pd.read_csv(PROCESSED_DATA_DIR / 'yearly_aggregated.csv', index_col=0, parse_dates=True)
df_monthly = pd.read_csv(PROCESSED_DATA_DIR / 'monthly_aggregated.csv', index_col=0, parse_dates=True)

print(f"Loaded Ulaanbaatar data: {len(df_yearly)} years")

## 1. Load External Data

### Global Temperature Data

Download global temperature anomaly data from:
- NASA GISS: https://data.giss.nasa.gov/gistemp/
- NOAA: https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/global/time-series
- Berkeley Earth: http://berkeleyearth.org/data/

Save to: `data/external/global_temperature.csv`

In [ ]:
# Example: Load global temperature data
# Adjust column names based on your data source

try:
    df_global_temp = pd.read_csv(EXTERNAL_DATA_DIR / 'global_temperature.csv')
    print("Global temperature data loaded!")
    print(df_global_temp.head())
except FileNotFoundError:
    print("⚠️ Global temperature data not found.")
    print("Please download from sources listed above and save to data/external/global_temperature.csv")
    df_global_temp = None

### World Population Data

Download from:
- World Bank: https://data.worldbank.org/indicator/SP.POP.TOTL
- UN: https://population.un.org/wpp/Download/Standard/CSV/

Save to: `data/external/world_population.csv`

In [ ]:
# Load world population data
try:
    df_population = pd.read_csv(EXTERNAL_DATA_DIR / 'world_population.csv')
    print("Population data loaded!")
    print(df_population.head())
except FileNotFoundError:
    print("⚠️ Population data not found.")
    print("Please download and save to data/external/world_population.csv")
    df_population = None

### Russia & China Economic Data

Download GDP, emissions, industrial production from:
- World Bank: https://data.worldbank.org/country
- IMF: https://www.imf.org/en/Data

Save to: `data/external/russia_china_economic.csv`

In [ ]:
# Load Russia & China economic data
try:
    df_economics = pd.read_csv(EXTERNAL_DATA_DIR / 'russia_china_economic.csv')
    print("Economic data loaded!")
    print(df_economics.head())
except FileNotFoundError:
    print("⚠️ Economic data not found.")
    print("Please download and save to data/external/russia_china_economic.csv")
    df_economics = None

## 2. Correlation with Global Temperature

Compare Ulaanbaatar warming with global warming trends.

In [ ]:
if df_global_temp is not None:
    # Analyze correlation with global temperature
    # Adjust column names based on your data
    global_corr = analyze_global_temperature_correlation(
        df_yearly,
        df_global_temp,
        ub_col='temp_mean',
        global_col='global_temp'  # Adjust this column name
    )
    
    # Plot comparison
    plot_comparison_with_global(
        df_yearly,
        df_global_temp,
        ub_col='temp_mean',
        global_col='global_temp',
        title='Ulaanbaatar vs Global Temperature',
        save_path=FIGURES_DIR / '20_ub_vs_global.png'
    )
    plt.show()
else:
    print("⚠️ Skipping global temperature analysis - data not available")

## 3. Correlation with Population

In [ ]:
if df_population is not None:
    # Analyze correlation with population
    pop_corr = analyze_population_correlation(
        df_yearly,
        df_population,
        temp_col='temp_mean',
        pop_col='population'  # Adjust column name
    )
    
    # Plot correlation
    merged = pop_corr['merged_data']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Time series
    ax1_twin = ax1.twinx()
    ax1.plot(merged.index, merged['temp_mean'], 'b-', label='Temperature')
    ax1_twin.plot(merged.index, merged['population'], 'r-', label='Population')
    ax1.set_xlabel('Year')
    ax1.set_ylabel('Temperature (°C)', color='b')
    ax1_twin.set_ylabel('Population', color='r')
    ax1.set_title('Temperature vs Population Over Time')
    
    # Scatter plot
    ax2.scatter(merged['population'], merged['temp_mean'], alpha=0.6)
    ax2.set_xlabel('Population')
    ax2.set_ylabel('Temperature (°C)')
    ax2.set_title('Temperature vs Population Correlation')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '21_temp_vs_population.png', dpi=300)
    plt.show()
else:
    print("⚠️ Skipping population analysis - data not available")

## 4. Correlation with Economic Indicators

In [ ]:
if df_economics is not None:
    # Analyze correlation with economic indicators
    # Adjust column names based on your data
    econ_cols = ['russia_gdp', 'china_gdp', 'russia_emissions', 'china_emissions']
    
    econ_corr = analyze_economic_correlation(
        df_yearly,
        df_economics,
        temp_col='temp_mean',
        econ_cols=econ_cols
    )
    
    # Create correlation summary
    print("\nEconomic Indicator Correlations:")
    for indicator, results in econ_corr.items():
        if indicator != 'merged_data':
            corr = results['correlation']
            print(f"\n{indicator}:")
            print(f"  Correlation: {corr['correlation']:.4f}")
            print(f"  P-value: {corr['p_value']:.6f}")
            print(f"  Significant: {corr['significant']}")
else:
    print("⚠️ Skipping economic analysis - data not available")

## 5. Comprehensive Correlation Matrix

In [ ]:
# Create combined dataset with all variables
# This will depend on what external data you have loaded

# Example:
# combined = df_yearly[['temp_mean']].copy()
# if df_global_temp is not None:
#     combined = pd.merge(combined, df_global_temp[['year', 'global_temp']], 
#                        left_on=combined.index.year, right_on='year', how='left')
# ... merge other datasets

# corr_matrix = calculate_correlation_matrix(combined)
# plot_correlation_matrix(
#     corr_matrix,
#     title='Comprehensive Correlation Matrix',
#     save_path=FIGURES_DIR / '22_correlation_matrix.png'
# )
# plt.show()

print("Create your own correlation matrix based on available external data!")

## 6. Cross-Correlation Analysis

In [ ]:
# Example: Cross-correlation with time lags
# This can reveal delayed relationships

if df_global_temp is not None:
    # Calculate cross-correlation
    merged = merge_external_data(df_yearly, df_global_temp, on='year')
    
    cross_corr = calculate_cross_correlation(
        merged['temp_mean'],
        merged['global_temp'],
        max_lag=10
    )
    
    # Plot cross-correlation
    plot_cross_correlation(
        cross_corr,
        'Ulaanbaatar Temp',
        'Global Temp',
        title='Cross-Correlation Analysis',
        save_path=FIGURES_DIR / '23_cross_correlation.png'
    )
    plt.show()
else:
    print("⚠️ Skipping cross-correlation - global data not available")

## 7. Granger Causality Tests

In [ ]:
# Test if external factors "Granger-cause" temperature changes
# This tests whether past values of X help predict future values of Y

if df_economics is not None:
    merged = merge_external_data(df_yearly, df_economics, on='year')
    
    # Example: Does China GDP Granger-cause temperature?
    gc_result = granger_causality_test(
        merged,
        'china_gdp',  # Adjust column name
        'temp_mean',
        max_lag=5
    )
    
    if gc_result:
        print("\nGranger Causality Results:")
        print(f"Best lag: {gc_result['best_lag']}")
        print(f"P-value at best lag: {gc_result['min_p_value']:.6f}")
        print(f"Significant: {gc_result['significant']}")
else:
    print("⚠️ Skipping Granger causality - economic data not available")

## Summary

This notebook provides a framework for:
- ✅ Comparing Ulaanbaatar temperature with global trends
- ✅ Analyzing correlation with population growth
- ✅ Testing relationships with economic indicators
- ✅ Performing cross-correlation analysis
- ✅ Testing for Granger causality

**Data Requirements:**
You need to obtain and format external data from the sources listed in the README.

**Key Findings:**
- Document your findings based on the external data you analyze
- Compare warming rates between Ulaanbaatar and global averages
- Identify significant correlations with external factors

**Next Steps:**
- Compile final report with all findings
- Create comprehensive visualizations
- Draw conclusions about climate change impacts on Mongolia